# Ingest - Store Vectors - Query

In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

## Ingest

In [2]:
# pip install google-cloud-aiplatform

from vertexai.language_models import TextEmbeddingModel

model = TextEmbeddingModel.from_pretrained("text-embedding-005")

def embed(texts: list[str]) -> list[list[float]]:
    embeddings = model.get_embeddings(texts)
    return [e.values for e in embeddings]

# Example: chunk a document into ~500-char pieces, then embed
chunks = ["GCP offers managed Kubernetes via GKE...", "BigQuery is serverless..."]
vectors = embed(chunks)  # shape: (2, 768)

## Store

In [3]:
ALREADY_GENERATED_INDEX = '' 
# ALREADY_GENERATED_INDEX = '7066578823902396416'

In [4]:
from google.cloud import aiplatform

aiplatform.init(project="moodels-educ", location="europe-west6")

if not ALREADY_GENERATED_INDEX == '':
    old_index = aiplatform.MatchingEngineIndex('projects/279637354150/locations/europe-west6/indexes/' + ALREADY_GENERATED_INDEX)
    old_index.delete()

# Create an index (one-time)
index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name="my-rag-index",
    dimensions=768,
    approximate_neighbors_count=10,
    leaf_node_embedding_count=500,
    index_update_method="STREAM_UPDATE",
)

# Upsert embeddings (can also batch-import from GCS)
index.upsert_datapoints(
    datapoints=[
        {"datapoint_id": f"chunk_{i}", "feature_vector": vec}
        for i, vec in enumerate(vectors)
    ]
)

Creating MatchingEngineIndex
Create MatchingEngineIndex backing LRO: projects/279637354150/locations/europe-west6/indexes/8194730530558705664/operations/3246085429904015360
MatchingEngineIndex created. Resource name: projects/279637354150/locations/europe-west6/indexes/8194730530558705664
To use this MatchingEngineIndex in another session:
index = aiplatform.MatchingEngineIndex('projects/279637354150/locations/europe-west6/indexes/8194730530558705664')
Upserting datapoints MatchingEngineIndex index: projects/279637354150/locations/europe-west6/indexes/8194730530558705664
MatchingEngineIndex index Upserted datapoints. Resource name: projects/279637354150/locations/europe-west6/indexes/8194730530558705664


resource name: projects/279637354150/locations/europe-west6/indexes/8194730530558705664

## Query

In [5]:
import vertexai
from vertexai.generative_models import GenerativeModel

# 1. Embed the user question
question = "What managed Kubernetes options does GCP have?"
[q_vector] = embed([question])

# 2. Retrieve top-k similar chunks from Vector Search
index_endpoint = aiplatform.MatchingEngineIndexEndpoint("projects/279637354150/locations/europe-west6/indexes/7066578823902396416")
matches = index_endpoint.find_neighbors(
    deployed_index_id="my_deployed_index",
    queries=[q_vector],
    num_neighbors=5,
)
retrieved_ids = [m.id for m in matches[0]]

# 3. Look up chunk text (you store id→text in Firestore / Cloud SQL / GCS)
context = "\n\n".join(chunk_store[chunk_id] for chunk_id in retrieved_ids)

# 4. Generate a grounded answer
vertexai.init(project="moodels-educ", location="europe-west6")
model = GenerativeModel("gemini-2.0-flash")

prompt = f"""Answer the question using only the context below.

Context:
{context}

Question: {question}
"""

response = model.generate_content(prompt)
print(response.text)

ValueError: Resource projects/279637354150/locations/europe-west6/indexes/7066578823902396416 is not a valid resource id.